### General Imports:


In [1]:
import os
import gymnasium as gym
from stable_baselines3.ppo import PPO
from stable_baselines3.ppo.policies import MlpPolicy as MLP_PPO
from stable_baselines3.common.monitor import Monitor
import matplotlib.pyplot as plt
import seaborn as sns
from netsim.gym_basic.envs import RMSA_ENV
from netsim.netSimPy import *
from sklearn import tree
from stable_baselines3.common.callbacks import EvalCallback
import numpy as np
import tensorflow as tf
import logging
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score

logging.getLogger("tensorflow").setLevel(logging.FATAL)
tf.__version__
from stable_baselines3.common.vec_env import (
    DummyVecEnv,
    is_vecenv_wrapped,
)

/Users/jbcedeno/Documents/projcts/multiband-gymnasium/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# FILES AND DIRECTORIES:
__file__ = "RMSA.ipynb"
absolutepath = os.path.abspath(__file__)
file_name = os.path.basename(os.path.abspath(__file__)).split(".")[0]
log_dir = f"./tmp/{file_name}/"
# SIMULATION PARAMS:
EPISODE_LENGTH = 1000
N_BLOCKS = 5
N_PATHS = 3
M_LAMBDA = 50000  # 500 Erlangs

networkPaths = "/Users/jbcedeno/Documents/projcts/multiband-gymnasium/networks/nsfnet"
network = Network(
    networkFileName=networkPaths + "/network.json",
    pathsFileName=networkPaths + "/routes.json",
    bitrateFilename=networkPaths + "/bitrates_c_bands.json",
)
generator = EventsGenerator(mLambda=M_LAMBDA)
sim_args = dict(eventsGenerator=generator, network=network)
simulator = NetworkSimulator(**sim_args)
# Building the environment:
env_args = dict(
    simulator=simulator,
    episode_length=EPISODE_LENGTH,
    j=N_BLOCKS,
    n_paths=N_PATHS,
)

env: RMSA_ENV = Monitor(gym.make("RMSA_ENV-v0", **env_args), log_dir)
eval_env: RMSA_ENV = Monitor(gym.make("RMSA_ENV-v0", **env_args), log_dir + "eval_env")


best_model_dir = f"./tmp/RMSA_27/best_model.zip"
expert = PPO.load(best_model_dir)

In [16]:
def generate_trajectories(n_timesteps, model, env, only_accepted=True):
    trajectories = []
    timesteps_count = 0
    state = None
    done = False
    observation = env.reset()[0]
    print("generate_trajectories", len(observation))

    while timesteps_count < n_timesteps:
        act, state = model.predict(
            observation,
            state=state,
            episode_start=done,
            deterministic=True,
        )
        traj = {"obs": observation, "action": act}
        observation, reward, done, truncated, info = env.step(act)
        print("generate_trajectories22", len(observation))
        traj["reward"] = reward
        if only_accepted:
            if reward > 0:
                trajectories.append(traj)
                timesteps_count += 1
        else:
            trajectories.append(traj)
            timesteps_count += 1
        if done:
            env.reset()
    return trajectories


def flatten_trajectories(traj):
    obs = [t["obs"] for t in traj]
    acts = [t["action"] for t in traj]
    rewards = [t["reward"] for t in traj]
    return obs, acts, rewards


class MixerPolicy:
    def __init__(self, expert, student, beta):
        self.student = student
        self.expert = expert
        self.beta = beta

    def set(self, expert, student, beta):
        self.student = student
        self.expert = expert
        self.beta = beta

    def predict(self, observation, state, episode_start, deterministic):
        model = np.random.choice(["A", "B"], p=[self.beta, 1 - self.beta])
        if model == "A":
            return self.expert.predict(observation, state, episode_start, deterministic)
        else:
            return self.student.predict([observation])[0], None


def get_expert_acts(expert, obs, env):
    acts = []
    state = None
    done = False
    env.reset()
    for o in obs:
        act, state = expert.predict(
            o,
            state=state,
            episode_start=done,
            deterministic=True,
        )
        acts.append(act)
    return acts


def linear_beta(start, end, steps):
    def call(n):
        return max(start + n * (end - start) / (steps - 1), end)

    return call

In [18]:
from sklearn import ensemble
from sklearn.linear_model import RidgeClassifier
from IPython.display import clear_output
import pickle as pkl
import pandas as pd
import json


def decode_action(act, j):
    idx_path = act // j
    idx_block = act % j
    return idx_path, idx_block


def get_route_from_action(acts, j):
    return [decode_action(a, j)[0] for a in acts]


def evaluate_student(student, n_timesteps):
    traj = generate_trajectories(n_timesteps, expert, env)
    x_test = [t["obs"] for t in traj]
    y_test = [t["action"] for t in traj]
    # y_test =  get_route_from_action(y_test, env.j)
    y_pred = student.predict(x_test)

    conf_matrix = confusion_matrix(y_test, y_pred)
    score = accuracy_score(y_test, y_pred)
    print(score, conf_matrix)


def train_with_dagger(
    expert,
    env,
    beta_func,
    n_rounds,
    timesteps_by_round,
    on_round_change=None,
    trajes_output_name=file_name + "_trajs.csv",
):
    student_policy = RidgeClassifier(positive=True)
    print(f"Starting round 1")
    print(f"Sampling new trajectories for round: {1} ")
    obs, acts, _rewards = flatten_trajectories(
        generate_trajectories(timesteps_by_round, expert, env)
    )

    # routes = get_route_from_action(acts, env.j)
    student_policy.fit(obs, acts)
    evaluate_student(student_policy, 1000)

    for r in range(1, n_rounds):
        print(f"Starting round {r+1}")
        mixer = MixerPolicy(expert, student_policy, beta=beta_func(r))
        print(f"Sampling new trajectories for round: {r+1} ")
        obs2, _acts, _rewards = flatten_trajectories(
            generate_trajectories(timesteps_by_round, mixer, env)
        )
        acts2 = get_expert_acts(expert, obs2, env)

        # extend and update:
        shuffle = np.arange(len(obs2))
        np.random.shuffle(shuffle)
        obs2 = [obs2[s] for s in shuffle]
        acts2 = [acts2[s] for s in shuffle]
        obs.extend(obs2)
        acts.extend(acts2)

        # print(f"Saving trajectories for round: {r+1} ")
        # df = pd.DataFrame({'x':[json.dumps(o.tolist(),separators=(',', ':')) for o in obs2], 'y':acts2})
        # df.to_csv(trajes_output_name,
        #             mode="a",
        #             index=False,
        #             header=False
        #         )
        print(f"Fitting student for round: {r+1} ")
        # clear_output(wait=True)
        print("OBS", len(obs[0]))
        print("acts", acts)
        student_policy.fit(obs, acts)
        evaluate_student(student_policy, 1000)

    return student_policy

In [19]:
# ELIMINR:
n_steps = 2
beta_func = linear_beta(1, 0, n_steps)
student2 = train_with_dagger(expert, env, beta_func, n_steps, timesteps_by_round=500)

importances_sk = student2.coef_[0]
# importance = pd.DataFrame(importances_sk)
# importance.to_csv(f"{output_dir}features_importance_heuristic.csv")
print(importances_sk)

Starting round 1
Sampling new trajectories for round: 1 
generate_trajectories 1099
generate_trajectories22 1099
generate_trajectories22 1099
generate_trajectories22 1099
generate_trajectories22 1099
generate_trajectories22 1099
generate_trajectories22 1099
generate_trajectories22 1099
generate_trajectories22 1099
generate_trajectories22 1099
generate_trajectories22 1099
generate_trajectories22 1099
generate_trajectories22 1099
generate_trajectories22 1099
generate_trajectories22 1099
generate_trajectories22 1099
generate_trajectories22 1099
generate_trajectories22 1099
generate_trajectories22 1099
generate_trajectories22 1099
generate_trajectories22 1099
generate_trajectories22 1099
generate_trajectories22 1099
generate_trajectories22 1099
generate_trajectories22 1099
generate_trajectories22 1099
generate_trajectories22 1099
generate_trajectories22 1099
generate_trajectories22 1099
generate_trajectories22 1099
generate_trajectories22 1099
generate_trajectories22 1099
generate_trajecto

In [ ]:
n_steps = 8
beta_func = linear_beta(1, 0, n_steps)
student = train_with_dagger(expert, env, beta_func, n_steps, timesteps_by_round=300000)

### Saving the agent


In [ ]:
pickle_out = open("student_LR.pkl", "wb")
pkl.dump(student, pickle_out)
pickle_out.close()
# pickle_out = open("student_LR.pkl","rb")
# student = pkl.load(pickle_out)
# pickle_out.close()

In [ ]:
def get_actions_count(y_test):
    y1_test = [0] * 15
    for yv in y_test:
        y1_test[yv] += 1
    return y1_test

### Global accuracy


In [ ]:
def get_metrics(matrix):
    # Calcula métricas
    precision = np.diag(matrix) / np.sum(matrix, axis=0)
    recall = np.diag(matrix) / np.sum(matrix, axis=1)
    specificity = np.diag(matrix) / (np.sum(matrix, axis=1) - np.diag(matrix))
    f1_score = 2 * (precision * recall) / (precision + recall)

    # Imprime las métricas
    for i in range(len(precision)):
        print(f"Clase {i + 1}:")
        print(f"  Precision: {precision[i]:.4f}")
        print(f"  F1-score: {f1_score[i]:.4f}")
        print(f"  Sensibilidad (Recall): {recall[i]:.4f}")
        print(f"  Especificidad: {specificity[i]:.4f}\n")

In [ ]:
traj = generate_trajectories(200000, expert, env)
x_test = [t["obs"] for t in traj]
y_test = [int(t["action"]) for t in traj]

# y_pred = predict(x_test,student, env)
y_pred = student.predict(x_test)
score = accuracy_score(y_test, y_pred)
print(score)
print(f1_score(y_test, y_pred, average="weighted"))
conf_matrix = confusion_matrix(y_test, y_pred, labels=[i for i in range(15)])
print(conf_matrix)

In [ ]:
get_metrics(conf_matrix)

### Inter-path accuracy


In [ ]:
n_conf_matrix = np.zeros(shape=(3, 3))
for ir, row in enumerate(conf_matrix):
    for ic, col in enumerate(row):
        n_conf_matrix[ir // 5, ic // 5] += conf_matrix[ir][ic]


def score(matrix):
    total = sum(matrix.reshape(-1, 1))
    correct = np.trace(matrix)
    return correct / total


s = score(n_conf_matrix)
print(s[0])
print(n_conf_matrix)

In [ ]:
# Matriz de confusión proporcionada
confusion_matrix = np.array(n_conf_matrix)

get_metrics(confusion_matrix)

### Inter-spectrum accuracy


In [ ]:
n2_conf_matrix = np.zeros(shape=(5, 5))
for ir, row in enumerate(conf_matrix):
    for ic, col in enumerate(row):
        n2_conf_matrix[ir % 5, ic % 5] += conf_matrix[ir][ic]


def score(matrix):
    total = sum(matrix.reshape(-1, 1))
    correct = np.trace(matrix)
    return correct / total


s = score(n2_conf_matrix)
print(s[0])
print(n2_conf_matrix)

In [ ]:
# Matriz de confusión proporcionada
confusion_matrix = np.array(n2_conf_matrix)

get_metrics(confusion_matrix)

In [ ]:
def evaluate_bp_student(student, n_timesteps, mLambda):
    timesteps_count = 0
    state = None
    done = False
    env.reset(options={"lambda": mLambda})
    observation = env.reset()[0]
    accepted_requests = 0

    while timesteps_count < n_timesteps:
        # act = predict([observation], student, env)[0]
        act = student.predict([observation])[0]
        observation, reward, done, truncated, info = env.step(act)
        if reward == 1:
            accepted_requests += 1
        if done:
            env.reset()
        timesteps_count += 1
    return 1 - accepted_requests / n_timesteps


def evaluate_expert(expert, n_timesteps, mLambda):
    timesteps_count = 0
    state = None
    done = False
    env.reset(options={"lambda": mLambda})
    observation = env.reset()
    accepted_requests = 0

    while timesteps_count < n_timesteps:
        act, state = expert.predict(
            observation,
            state=state,
            episode_start=done,
            deterministic=True,
        )
        observation, reward, done, truncated, info = env.step(act)
        if reward == 1:
            accepted_requests += 1
        if done:
            env.reset()
        timesteps_count += 1
    return 1 - accepted_requests / n_timesteps


# loads = [1000, 1200, 1400, 1600, 1800, 2000,]
# results = {
#     "model": [],
#     "student": [],
# }
# N_EVALUATIONS = 400000

# for l in loads:
#     env.setAllocatorFunc(None)
#     # agent_model, _ = evaluate_policy(env, n_eval_episodes=N_EVALUATION_EPISODES, model = model, return_dataframe=False, return_episode_rewards=False, seed=1)
#     bp = evaluate_expert(expert, N_EVALUATIONS, l)
#     print("model",bp)
#     results["model"].append(bp)

#     bp = evaluate_bp_student(student, N_EVALUATIONS, l)
#     print("student",bp)
#     results["student"].append(bp)

In [ ]:
loads = [5000, 10000, 15000, 20000, 25000, 30000, 35000, 40000]
N_EVALUATIONS = 1000000
lr_bp = []

for l in loads:
    bp = evaluate_bp_student(student, N_EVALUATIONS, l)
    lr_bp.append(round(bp, 7))
print(lr_bp)

### features importance


In [ ]:
import pandas as pd

base_output_dir_features = f"./plots/{__file__}/features_importance_heuristic/"

output_dir = base_output_dir_features
os.makedirs(output_dir, exist_ok=True)

# print("lolo",student.coef_[0])
# print("lolo",len(student.get_params()))

importances_sk = student.coef_[0]
importance = pd.DataFrame(importances_sk)
importance.to_csv(f"{output_dir}features_importance_heuristic.csv")
print(importances_sk)
plt.figure()
plt.bar([x for x in range(len(importances_sk))], importances_sk)
plt.xlabel("Feature")
plt.ylabel("Importance")
plt.title(f"Features importance NSFNet")
plt.savefig(output_dir + "NSFNet.png")
plt.show()